# Modélisation : effet des médailles olympiques sur le nombre de licenciés

**Question** : les performances olympiques (médailles) ont-elles un impact mesurable sur l’évolution du nombre de licenciés, et comment se compare cet effet à la dynamique “inertielle” du nombre de licenciés ?

Nous mettons en œuvre deux approches complémentaires :

1) **Modèle économétrique (principal)** : estimation d’un effet moyen des médailles sur la **croissance** des licenciés, avec effets fixes sport et année, et erreurs robustes clusterisées par sport.

2) **Modèle prédictif (secondaire, non causal)** : modélisation du **niveau** de licenciés pour produire des trajectoires observé vs prédit par sport et quantifier le rôle de l’inertie.


## 1) Imports + accès au package


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)

# imports package
from model.feature_engineering import build_lic_features, merge_medals
from model.modele_econo import run_econo_model
from model.modele_predictif import train_predictive_model, plot_pred_sport


Nous utilisons un package (`model/`) pour garantir un code propre et reproductible : le notebook se limite à appeler des fonctions et à commenter les résultats.


## 2) Construction du panel final df_model_full

In [ ]:
# df_lic : données licences individuelles (avec age, sexe, dep, etc.)
# df_med : table JO par sport (médailles 2016/2020/2024)
df_features = build_lic_features(df_lic)

# Merge sur le JO de référence : on obtient total_medailles correspondant au "dernier JO" pour chaque année
df_model_full = merge_medals(df_features, df_med)

print("df_model_full shape:", df_model_full.shape)
df_model_full.head()


Nous construisons un panel sport–année qui regroupe :
- le nombre de licenciés (variable d’intérêt),
- des variables structurelles (composition par sexe, âge, dispersion géographique),
- et une mesure des performances olympiques (`total_medailles`) associée au dernier JO de référence (`jo_ref`).

Ce panel permet une analyse économétrique (effets fixes + dynamique) ainsi qu’une modélisation prédictive.


## 3) Sanity checks (anti-bugs + anti-incohérences)

In [ ]:
# unicité sport-année
print("Duplicats sport-année :", df_model_full.duplicated(["code_sport","annee"]).sum())

# vérifier que total_medailles est constant pour un sport donné à JO donné
tmp = df_model_full.groupby(["code_sport","jo_ref"])["total_medailles"].nunique().reset_index()
print("Max nunique(total_medailles) par (sport, jo_ref):", tmp["total_medailles"].max())

# distribution jo_ref par année
df_model_full.groupby(["annee","jo_ref"]).size().head(20)


Ces contrôles vérifient :
- l’unicité des observations (sport–année),
- la cohérence de l’association “année → dernier JO” (`jo_ref`),
- et l’absence de double comptage des médailles (une seule valeur `total_medailles` par sport et JO).


## 4) Spécification économétrique

# Modèle principal : effet des médailles sur la croissance des licenciés (panel)

Nous estimons :

\[
\Delta \log(1+Lic_{s,t}) = \beta \cdot Med_{s,JO} + \alpha_s + \gamma_t + \varepsilon_{s,t}
\]

- \(\Delta \log(1+Lic_{s,t})\) : croissance annuelle approximative du nombre de licenciés (stabilisation de variance, comparabilité entre sports).
- \(\alpha_s\) : **effets fixes sport** (popularité structurelle, culture de pratique, taille de base).
- \(\gamma_t\) : **effets fixes année** (chocs communs : Covid, tendances globales, politiques publiques).
- Erreurs **clusterisées par sport** pour tenir compte de la corrélation intra-sport dans le temps.

Interprétation : si \(\beta\) est petit, \(100\beta\) ≈ variation en points de pourcentage de la croissance annuelle associée à une médaille supplémentaire.


## 5) Estimation du modèle économétrique

In [ ]:
results_econo = run_econo_model(
    df_model_full,
    medals_col="total_medailles",
    start_year=2017,
    end_year=2024,
    controls=["part_femmes", "age_mean", "nb_departements_actifs"]  # optionnel
)

print(results_econo.summary())


## 6) Interprétation chiffrée du coefficient

In [ ]:
beta = float(results_econo.params["med_last"])
ci = results_econo.conf_int().loc["med_last"]

print(f"beta = {beta:.6f}")
print(f"Interprétation: +1 médaille -> ~ {100*beta:.2f}% de croissance annuelle (approx.)")
print(f"IC 95%: [{100*ci[0]:.2f}%, {100*ci[1]:.2f}%]")


Ce coefficient mesure un **effet moyen** (toutes disciplines confondues), conditionnel aux effets fixes sport et année.
Il ne s’agit pas d’un effet “par sport” mais d’un impact moyen des performances olympiques sur la dynamique des licenciés.


## Preuve 1 — Décomposition de la prédiction (ablation)

Objectif : quantifier le rôle de l’inertie versus le rôle des médailles en prédiction hors échantillon.
Nous comparons 3 modèles prédictifs (sur 2022–2024) :

- M0 : inertie seule (lag)
- M1 : médailles seules
- M2 : inertie + médailles

Important : ceci est un exercice prédictif, pas causal. Un bon R² des médailles seules peut refléter la popularité structurelle des sports.


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

def eval_ablation_models(df_model_full, train_years=(2017, 2020), test_years=(2022, 2024),
                         medals_col="total_medailles", alpha=1.0):
    df = df_model_full.sort_values(["code_sport","annee"]).copy()
    df["y"] = np.log1p(df["nb_licencies"])
    df["log_lag1"] = df.groupby("code_sport")["y"].shift(1)
    df["trend"] = df["annee"] - df["annee"].min()
    df["med_last"] = df[medals_col]
    df = df.dropna(subset=["log_lag1"]).copy()

    train_df = df[df["annee"].between(*train_years)].copy()
    test_df  = df[df["annee"].between(*test_years)].copy()

    def fit_predict(feats):
        Xtr = pd.get_dummies(train_df[["code_sport"] + feats], columns=["code_sport"], drop_first=True).astype("float32")
        Xte = pd.get_dummies(test_df[["code_sport"] + feats], columns=["code_sport"], drop_first=True).astype("float32")
        Xte = Xte.reindex(columns=Xtr.columns, fill_value=0).astype("float32")

        ytr = train_df["y"].astype("float32").values
        yte = test_df["y"].astype("float32").values

        m = Ridge(alpha=alpha)
        m.fit(Xtr, ytr)
        pred = m.predict(Xte)

        return float(r2_score(yte, pred)), float(mean_absolute_error(np.expm1(yte), np.expm1(pred)))

    rows = []
    for name, feats in [
        ("M0: inertie seule", ["log_lag1"]),
        ("M1: médailles seules", ["med_last"]),
        ("M2: inertie + médailles", ["log_lag1","med_last"]),
    ]:
        r2, mae = fit_predict(feats)
        rows.append({"modele": name, "R2_log": r2, "MAE_niveau": mae})
    return pd.DataFrame(rows)

ablation = eval_ablation_models(df_model_full)
ablation


Interprétation attendue :
- Si M0 est très performant, cela indique une forte inertie de la série.
- Si M2 n’améliore que marginalement M0, les médailles ont un pouvoir prédictif marginal faible dans ce cadre.
- Si M1 semble “bon”, cela peut refléter une corrélation avec la popularité structurelle (pas causal).


## Preuve 2 — Poids de l’inertie (lecture des coefficients Ridge)

Objectif : illustrer quantitativement que le lag du niveau de licenciés est le principal déterminant prédictif.


In [ ]:
def ridge_coefficients(df_model_full, train_years=(2017, 2020), medals_col="total_medailles", alpha=1.0):
    df = df_model_full.sort_values(["code_sport","annee"]).copy()
    df["y"] = np.log1p(df["nb_licencies"])
    df["log_lag1"] = df.groupby("code_sport")["y"].shift(1)
    df["trend"] = df["annee"] - df["annee"].min()
    df["med_last"] = df[medals_col]
    df = df.dropna(subset=["log_lag1"]).copy()

    train_df = df[df["annee"].between(*train_years)].copy()
    feats = ["log_lag1", "trend", "med_last"]

    X = pd.get_dummies(train_df[["code_sport"] + feats], columns=["code_sport"], drop_first=True).astype("float32")
    y = train_df["y"].astype("float32").values

    m = Ridge(alpha=alpha)
    m.fit(X, y)

    coef = pd.Series(m.coef_, index=X.columns).sort_values(key=lambda s: s.abs(), ascending=False)
    return coef

coef = ridge_coefficients(df_model_full)
coef.head(15)


Si `log_lag1` apparaît comme le coefficient dominant, cela confirme empiriquement que la dynamique des licenciés est principalement inertielle.


## Preuve 3 — Apport marginal des médailles dans le modèle économétrique

Objectif : tester si l’ajout de la variable médailles améliore significativement l’explication de la croissance,
une fois contrôlés les effets fixes sport et année.
Nous comparons :
- modèle restreint : sans médailles
- modèle complet : avec médailles


In [ ]:
import statsmodels.formula.api as smf

def test_medals_incremental(df_model_full, medals_col="total_medailles", start=2017, end=2024):
    dfp = df_model_full.sort_values(["code_sport","annee"]).copy()
    dfp["log_lic"] = np.log1p(dfp["nb_licencies"])
    dfp["dlog_lic"] = dfp.groupby("code_sport")["log_lic"].diff(1)
    dfp = dfp[dfp["annee"].between(start, end)].dropna(subset=["dlog_lic"]).copy()
    dfp["med_last"] = dfp[medals_col]

    controls = [c for c in ["part_femmes","age_mean","nb_departements_actifs"] if c in dfp.columns]
    ctrl = (" + " + " + ".join(controls)) if controls else ""

    f0 = f"dlog_lic ~ C(code_sport) + C(annee){ctrl}"
    f1 = f"dlog_lic ~ med_last + C(code_sport) + C(annee){ctrl}"

    m0 = smf.ols(f0, data=dfp).fit(cov_type="cluster", cov_kwds={"groups": dfp["code_sport"]})
    m1 = smf.ols(f1, data=dfp).fit(cov_type="cluster", cov_kwds={"groups": dfp["code_sport"]})

    wald = m1.wald_test("med_last = 0")
    return m0, m1, wald

m0, m1, wald = test_medals_incremental(df_model_full)
print(wald)
print("beta medals:", m1.params["med_last"])


- Si la p-value du test est faible, cela suggère que les médailles contribuent statistiquement à expliquer la croissance,
  au-delà des effets fixes.
- Cela ne “prouve” pas une causalité parfaite, mais renforce la crédibilité de l’association conditionnelle estimée.


## Modèle prédictif (complément descriptif, non causal)

Ce modèle n’a pas vocation à identifier un effet causal des médailles.
Il est utilisé pour :
- produire des trajectoires observé vs prédit par sport,
- quantifier l’inertie (poids du passé),
- et illustrer les limites empiriques de l’identification causale.


In [ ]:
sec_model, perf = train_predictive_model(df_model_full)
perf


Une performance élevée est attendue car le modèle inclut une variable d’inertie (lag), qui explique une grande partie de la dynamique des licenciés.


## 12) Graphiques par sport

In [ ]:
plot_pred_sport("HAN")
plot_pred_sport("ATH")


Les courbes montrent que la variation du nombre de licenciés est largement inertielle.
Les médailles jouent un rôle plus marginal, ce qui est cohérent avec les résultats du modèle économétrique.


## Conclusion (modélisation)

- Le modèle économétrique (panel FE + erreurs cluster) suggère un effet moyen des médailles sur la croissance des licenciés, quantitativement modeste mais potentiellement significatif.
- Les “preuves” complémentaires (ablation, coefficients Ridge, test d’apport marginal) indiquent que la dynamique des licenciés est fortement dominée par l’inertie et la popularité structurelle des sports.
- Le modèle prédictif est utilisé à des fins descriptives et de visualisation, sans interprétation causale.

Ces résultats convergent : les JO peuvent contribuer à la dynamique des licenciés, mais ne sont pas le déterminant principal des trajectoires observées.
